# 01 · RoBERTa-base + span head

Pipeline: §2 Extractive model in `docs/pipeline.md`.

Fully fine-tune RoBERTa-base with a span head on the training split, then predict on the dev set.

- **Reads:** `data/splits/train_val_articles.json`
- **Writes:** `data/predictions/roberta.jsonl`; checkpoint in `checkpoints/roberta/` (Drive only)
- **GPU:** yes (T4 is enough)
- **Owner:** TBD


## Setup


In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import os
from pathlib import Path

# Colab: shared Drive folder. Kaggle: working dir. Local: the repo itself.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/hlt-qa")
except ImportError:
    ROOT = Path("/kaggle/working/hlt-qa") if Path("/kaggle").exists() else Path("..")

SPLITS_DIR = ROOT / "data" / "splits"
PREDICTIONS_DIR = ROOT / "data" / "predictions"
METRICS_DIR = ROOT / "data" / "metrics"
JUDGE_DIR = ROOT / "data" / "judge"
ANNOTATION_DIR = ROOT / "data" / "annotation"
TABLES_DIR = ROOT / "results" / "tables"
FIGURES_DIR = ROOT / "results" / "figures"
CHECKPOINTS_DIR = ROOT / "checkpoints"  # not committed to git

for d in [SPLITS_DIR, PREDICTIONS_DIR, METRICS_DIR, JUDGE_DIR, ANNOTATION_DIR,
          TABLES_DIR, FIGURES_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SPLIT_FILE = SPLITS_DIR / "train_val_articles.json"
assert SPLIT_FILE.exists(), f"{SPLIT_FILE} not found: copy train_val_articles.json there (or run 00_data_splits.ipynb)."
print("ROOT =", ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT = /content/drive/MyDrive/hlt-qa


## Settings


In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

MODELS = {"roberta": "roberta-base", "bert": "bert-base-uncased"}
SYSTEM_ID = "roberta"  # change to "bert" for the BERT baseline
MODEL_NAME = MODELS[SYSTEM_ID]
DATASET = "rajpurkar/squad"

# Sliding windows for long passages (standard values for SQuAD, same for RoBERTa and BERT).
# The model reads at most 512 tokens; 384 is a cheaper choice (attention cost grows
# quadratically) and still fits almost every SQuAD question + passage (~1% get split).
# Longer passages are split into windows that share STRIDE tokens, so an answer cut at
# the border of one window appears whole in the next one.
MAX_LENGTH = 384  # tokens per window (question + passage), must be <= 512
STRIDE = 128      # overlap between consecutive windows

# Training (starting point)
LEARNING_RATE = 3e-5
EPOCHS = 2
BATCH_SIZE = 16
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
EVAL_BATCH_SIZE = 64
EVAL_STEPS = 1000  # evaluate on val and save a checkpoint every N steps

# Quick test: train on a few windows to check the whole pipeline. Set to False for the real run.
QUICK_TEST = True

# Span decoding (standard values for SQuAD; they only change decoding, no re-training needed).
# For each window we try the N_BEST best start tokens x N_BEST best end tokens (20 x 20 = 400
# pairs): the best valid span is almost always among them, a larger N_BEST is only slower.
# MAX_ANSWER_LENGTH drops implausibly long spans: SQuAD answers are short (a few words),
# so 30 tokens keeps practically all of them.
N_BEST = 20
MAX_ANSWER_LENGTH = 30  # tokens

OUTPUT_FILE = PREDICTIONS_DIR / f"{SYSTEM_ID}.jsonl"
CHECKPOINT_DIR = CHECKPOINTS_DIR / SYSTEM_ID

Device: cuda


## 2. Data

### 2.1 Load SQuAD and the saved split

The split file lists the article titles of train and validation. We keep the training questions whose article is in each list. The dev set is the official SQuAD validation split (our test set), used as it is.

In [ ]:
import json
from datasets import load_dataset

split = json.loads(SPLIT_FILE.read_text(encoding="utf-8"))
train_titles, val_titles = set(split["train"]), set(split["val"])

raw = load_dataset(DATASET)
train_ds = raw["train"].filter(lambda ex: ex["title"] in train_titles)
val_ds = raw["train"].filter(lambda ex: ex["title"] in val_titles)
dev_ds = raw["validation"]  # official dev = our test set

# Sanity checks
assert not (train_titles & val_titles)
assert len(train_ds) + len(val_ds) == len(raw["train"])
assert set(dev_ds["title"]) == set(split["dev"]), "dev articles differ from the split."
print(f"train: {len(train_ds)}  val: {len(val_ds)}  dev:{len(dev_ds)}")

train: 83575  val: 4024  dev:10570


### 2.2 Check the answer positions

The labels we build later rely on `answer_start`. Here we verify that, for every training and validation question, the characters of the passage starting at `answer_start` are exactly the answer text.

In [ ]:
n_checked = n_multi = 0
for name, ds in [("train", train_ds), ("val", val_ds)]:
  for ex in ds:
    starts, texts = ex["answers"]["answer_start"], ex["answers"]["text"]
    n_multi += len(texts) > 1
    s, t = starts[0], texts[0]
    assert ex["context"][s:s + len(t)] == t, f"{name}: bad position in {ex['id']}"
    n_checked += 1

print(f"Checked {n_checked} questions: answer positions OK")
# In case we get questions with more than one answer -> we need to take only first one for training
print("Questions with more than one answer:", n_multi)

Checked 87599 questions: answer positions OK
Questions with more than one answer: 0


## 3. Preprocessing

### 3.1 Explore the tokenizer
Encode one question + passage. Look at the tokens, `sequence_ids` (0 = question, 1 = passage,
None = special) and `offset_mapping` (token -> characters of the original text).

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.is_fast, "we need a fast tokenizer to get offset_mapping"

ex = train_ds[0]
enc = tokenizer(ex["question"].lstrip(), ex["context"], return_offsets_mapping=True)

tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
seq_ids = enc.sequence_ids()
offsets = enc["offset_mapping"]

print("Question:", ex["question"])
print("Answer:  ", ex["answers"]["text"][0])
print("Number of tokens:", len(tokens), "\n")
print(f"{'i':>3}  {'token':<15} {'seq_id':<7} offsets")
for i in range(25):
    print(f"{i:>3}  {tokens[i]!r:<15} {str(seq_ids[i]):<7} {offsets[i]}")

Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Answer:   Saint Bernadette Soubirous
Number of tokens: 181 

  i  token           seq_id  offsets
  0  '<s>'           None    (0, 0)
  1  'To'            0       (0, 2)
  2  'Ġwhom'         0       (3, 7)
  3  'Ġdid'          0       (8, 11)
  4  'Ġthe'          0       (12, 15)
  5  'ĠVirgin'       0       (16, 22)
  6  'ĠMary'         0       (23, 27)
  7  'Ġallegedly'    0       (28, 37)
  8  'Ġappear'       0       (38, 44)
  9  'Ġin'           0       (45, 47)
 10  'Ġ18'           0       (48, 50)
 11  '58'            0       (50, 52)
 12  'Ġin'           0       (53, 55)
 13  'ĠL'            0       (56, 57)
 14  'our'           0       (57, 60)
 15  'des'           0       (60, 63)
 16  'ĠFrance'       0       (64, 70)
 17  '?'             0       (70, 71)
 18  '</s>'          None    (0, 0)
 19  '</s>'          None    (0, 0)
 20  'Arch'          1       (0, 4)
 21  'itect'         1       (4, 9)

### 3.2 Find the answer tokens by hand
For one example, use the offsets to find the first and last token of the answer from the
character positions (`answer_start`, `answer_start + len(text)`). Decode them: they must give back the answer.

In [ ]:
answer = ex["answers"]
start_char = answer["answer_start"][0]
end_char = start_char + len(answer["text"][0])

# first and last token of the passage (sequence id 1)
ctx_start = seq_ids.index(1)
ctx_end = len(seq_ids) - 1 - seq_ids[::-1].index(1)

# start token: last passage token that starts at or before start_char
start_tok = ctx_start
while start_tok <= ctx_end and offsets[start_tok][0] <= start_char:
    start_tok += 1
start_tok -= 1

# end token: first passage token that ends at or after end_char
end_tok = ctx_end
while end_tok >= ctx_start and offsets[end_tok][1] >= end_char:
    end_tok -= 1
end_tok += 1

print("Answer tokens:", start_tok, "->", end_tok)
print("Gold answer:  ", repr(answer["text"][0]))
print("Decoded:      ", repr(tokenizer.decode(enc["input_ids"][start_tok:end_tok + 1])))
print("From offsets: ", repr(ex["context"][offsets[start_tok][0]:offsets[end_tok][1]]))

Answer tokens: 135 -> 142
Gold answer:   'Saint Bernadette Soubirous'
Decoded:       ' Saint Bernadette Soubirous'
From offsets:  'Saint Bernadette Soubirous'


### 3.3 Training features
`prepare_train_features`: tokenize with windows (`MAX_LENGTH`, `STRIDE`) and label each window
with `start_positions` / `end_positions`. A window that does not fully contain the answer is
labelled with \<s\> (index 0).

In [ ]:
def prepare_train_features(examples, max_length=MAX_LENGTH, stride=STRIDE, keep_sample_idx=False):
    """Tokenize question + passage in windows and label each window with the answer tokens."""
    questions = [q.lstrip() for q in examples["question"]]
    enc = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",  # cut only the passage, never the question
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = enc.pop("overflow_to_sample_mapping")  # window -> question
    offset_mapping = enc.pop("offset_mapping")
    enc["start_positions"], enc["end_positions"] = [], []

    for i, offsets in enumerate(offset_mapping):
        answers = examples["answers"][sample_map[i]]
        start_char = answers["answer_start"][0]  # first answer only for training
        end_char = start_char + len(answers["text"][0])

        seq_ids = enc.sequence_ids(i)
        ctx_start = seq_ids.index(1)
        ctx_end = len(seq_ids) - 1 - seq_ids[::-1].index(1)

        # answer not fully inside this window -> label with <s> / [CLS] (index 0)
        if offsets[ctx_start][0] > start_char or offsets[ctx_end][1] < end_char:
            enc["start_positions"].append(0)
            enc["end_positions"].append(0)
            continue

        s = ctx_start
        while s <= ctx_end and offsets[s][0] <= start_char:
            s += 1
        e = ctx_end
        while e >= ctx_start and offsets[e][1] >= end_char:
            e -= 1
        enc["start_positions"].append(s - 1)
        enc["end_positions"].append(e + 1)

    if keep_sample_idx:  # only for the tests in 3.4
        enc["sample_idx"] = sample_map
    return enc

### 3.4 Test the labels
Run 3.3 on a few examples, decode the tokens between start and end and compare with the gold
answer. Check also a case with several windows (small `MAX_LENGTH`, then restore the real values).

In [ ]:
def norm(s):
    # ignore case and spaces: BERT-uncased lowercases, decode() can change spacing
    return "".join(s.lower().split())

def check_labels(ds, max_length=MAX_LENGTH, stride=STRIDE):
    batch = ds[:]
    feats = prepare_train_features(batch, max_length, stride, keep_sample_idx=True)
    n_ok = n_bad = n_zero = 0
    for i, ids in enumerate(feats["input_ids"]):
        s, e = feats["start_positions"][i], feats["end_positions"][i]
        if s == 0 and e == 0:
            n_zero += 1
            continue
        gold = batch["answers"][feats["sample_idx"][i]]["text"][0]
        decoded = tokenizer.decode(ids[s:e + 1])
        if norm(decoded) == norm(gold):
            n_ok += 1
        else:
            n_bad += 1
            if n_bad <= 5:
                print("  mismatch:", repr(gold), "->", repr(decoded))
    print(f"max_length={max_length}: {len(ds)} questions -> {len(feats['input_ids'])} windows | "
          f"ok: {n_ok}  mismatch: {n_bad}  answer not in window: {n_zero}")

sample = train_ds.select(range(300))
check_labels(sample)                            # real values
check_labels(sample, max_length=128, stride=32)  # small windows: many questions split in several windows

  mismatch: '19.7%' -> '19.7%).'
  mismatch: '39.1%' -> '39.1%)'
max_length=384: 300 questions -> 305 windows | ok: 298  mismatch: 2  answer not in window: 5
  mismatch: '19.7%' -> '19.7%).'
  mismatch: '39.1%' -> '39.1%)'
  mismatch: '39.1%' -> '39.1%)'
max_length=128: 300 questions -> 752 windows | ok: 355  mismatch: 3  answer not in window: 394


### 3.5 Evaluation features
`prepare_eval_features`: same tokenization, no labels. Keep `example_id` per window and the
character offsets only for passage tokens (None for the others), needed later for decoding.

In [ ]:
def prepare_eval_features(examples, max_length=MAX_LENGTH, stride=STRIDE):
    """Same windows as training, no labels. Keep example_id and passage offsets for decoding."""
    questions = [q.lstrip() for q in examples["question"]]
    enc = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = enc.pop("overflow_to_sample_mapping")
    enc["example_id"] = []

    for i in range(len(enc["input_ids"])):
        seq_ids = enc.sequence_ids(i)
        enc["example_id"].append(examples["id"][sample_map[i]])
        # None for question / special / padding tokens -> they can never be an answer
        enc["offset_mapping"][i] = [o if seq_ids[k] == 1 else None
                                    for k, o in enumerate(enc["offset_mapping"][i])]
    return enc

test = prepare_eval_features(dev_ds.select(range(5))[:])
print(len(test["input_ids"]), "windows for 5 questions")
print("example_id:", test["example_id"][:3])
print("offsets:", test["offset_mapping"][0][:20])

5 windows for 5 questions
example_id: ['56be4db0acb8001400a502ec', '56be4db0acb8001400a502ed', '56be4db0acb8001400a502ee']
offsets: [None, None, None, None, None, None, None, None, None, None, None, None, None, None, (0, 5), (6, 10), (11, 13), (14, 17), (18, 20), (21, 29)]


### 3.6 Apply to all sets
`.map(...)` on train, validation and dev. Check that the numbers of windows are consistent
(at least as many windows as questions).

In [ ]:
train_feats = train_ds.map(prepare_train_features, batched=True, remove_columns=train_ds.column_names)
val_feats = val_ds.map(prepare_train_features, batched=True, remove_columns=val_ds.column_names)  # with labels -> validation loss
val_eval_feats = val_ds.map(prepare_eval_features, batched=True, remove_columns=val_ds.column_names)  # for decoding
dev_feats = dev_ds.map(prepare_eval_features, batched=True, remove_columns=dev_ds.column_names)

for name, ds, feats in [("train", train_ds, train_feats), ("val", val_ds, val_feats),
                        ("val (eval)", val_ds, val_eval_feats), ("dev", dev_ds, dev_feats)]:
    assert len(feats) >= len(ds), f"{name}: fewer windows than questions"
    print(f"{name:<11} {len(ds):>6} questions -> {len(feats):>6} windows")

n_zero = sum(1 for s in train_feats["start_positions"] if s == 0)
print(f"\ntrain windows without the answer (label 0): {n_zero} ({n_zero / len(train_feats):.1%})")

train        83575 questions ->  84516 windows
val           4024 questions ->   4052 windows
val (eval)    4024 questions ->   4052 windows
dev          10570 questions ->  10790 windows

train windows without the answer (label 0): 765 (0.9%)


## 4. Model
Load the pre-trained encoder with a new **span head**: a linear layer (hidden size -> 2) that gives every token a *start* score and an *end* score. The encoder weights are pre-trained, the span head starts random (the warning about newly initialized `qa_outputs` weights is expected).

In [ ]:
from transformers import AutoModelForQuestionAnswering

torch.manual_seed(SEED)  # the span head is initialized randomly
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_NAME}: {n_params / 1e6:.1f}M parameters")
print("Span head:", model.qa_outputs)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForQuestionAnswering LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
qa_outputs.bias           | MISSING    | 
qa_outputs.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta-base: 124.1M parameters
Span head: Linear(in_features=768, out_features=2, bias=True)


## 5. Training
Full fine-tuning with the Hugging Face `Trainer`. Every `EVAL_STEPS` steps we compute the loss on the validation windows and save a checkpoint on Drive; at the end the checkpoint with the lowest validation loss is kept. If Colab disconnects, re-running this cell resumes from the last checkpoint.

With `QUICK_TEST = True` (Settings) it trains only on 2,000 windows, to check that everything works before the real run.

---
> **TODO: hyperparameter tuning.** The values in Settings are a standard starting point, not tuned. The training cell has to be changed to run a small search: one run per configuration, each in its own `run_dir`, recording the validation loss and the validation F1 (part 7), then keep the best configuration. Selection is done **only on the validation set**, never on dev (the test set).
>
> **To tune** (they have the largest effect on fine-tuning):
> - `LEARNING_RATE`: the most important one; try e.g. {1e-5, 2e-5, 3e-5, 5e-5}. Too high makes training unstable, too low under-fits in 2 epochs.
> - `EPOCHS`: {2, 3}. Look at the validation loss curve: if it still goes down at the end, one more epoch can help; if it goes up, the model over-fits.
> - `BATCH_SIZE` and `WARMUP_RATIO` (optional, smaller effect): e.g. batch {16, 32} (limited by the T4 memory), warmup {0.06, 0.1}. Learning rate and batch size interact, so tune them together if both are tuned.
>
> **Keep fixed** (and why):
> - `SEED`: same for every run, so the differences come from the hyperparameters and not from randomness.
> - `MAX_LENGTH` / `STRIDE`: input format, not a learning hyperparameter; 384/128 already fits ~99% of the questions.
> - `N_BEST` / `MAX_ANSWER_LENGTH`: decoding only; if needed they can be checked afterwards on validation without re-training.
> - `WEIGHT_DECAY`: standard 0.01, very small effect.
>
> **Cost and fairness:** one full run takes about 1 hour on a T4, so keep the grid small (or compare configurations on a subset of the training windows first). Use the same search for RoBERTa and BERT, so the comparison stays fair.

In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator
from transformers.trainer_utils import get_last_checkpoint

if QUICK_TEST:
    train_set = train_feats.shuffle(seed=SEED).select(range(2000))
    val_set = val_feats.select(range(500))
    run_dir = CHECKPOINT_DIR / "quick_test"
    import shutil
    shutil.rmtree(run_dir, ignore_errors=True)  # a quick test always starts from scratch
    eval_steps, epochs = 50, 1
else:
    train_set, val_set = train_feats, val_feats
    run_dir = CHECKPOINT_DIR / "run"
    eval_steps, epochs = EVAL_STEPS, EPOCHS

import math
total_steps = math.ceil(len(train_set) / BATCH_SIZE) * epochs
warmup_steps = int(WARMUP_RATIO * total_steps)  # warmup_ratio was removed in recent transformers
print(f"{len(train_set)} windows, {total_steps} steps, {warmup_steps} warmup steps")

args = TrainingArguments(
    output_dir=str(run_dir),
    learning_rate=LEARNING_RATE,
    num_train_epochs=epochs,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    warmup_steps=warmup_steps,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="linear",
    fp16=(DEVICE == "cuda"),       # mixed precision: faster on the T4
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    save_total_limit=2,            # keep only the best and the last checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=max(10, eval_steps // 10),
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_set,
    eval_dataset=val_set,
    data_collator=default_data_collator,
)

last_ckpt = get_last_checkpoint(str(run_dir)) if run_dir.exists() else None
if last_ckpt:
    print("Resuming from", last_ckpt)
trainer.train(resume_from_checkpoint=last_ckpt)

best_dir = run_dir / "best"
trainer.save_model(str(best_dir))
print("Best checkpoint:", trainer.state.best_model_checkpoint, "-> saved in", best_dir)

2000 windows, 125 steps, 12 warmup steps


Step,Training Loss,Validation Loss
50,3.306290,2.452065
100,2.193245,1.678427
125,1.621782,1.630246


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best checkpoint: /content/drive/MyDrive/hlt-qa/checkpoints/roberta/quick_test/checkpoint-125 -> saved in /content/drive/MyDrive/hlt-qa/checkpoints/roberta/quick_test/best


## 6. Decoding
The model gives every token of every window a *start* score and an *end* score. To get the answer text:
1. in each window take the `N_BEST` best start and end tokens;
2. drop impossible pairs: end before start, longer than `MAX_ANSWER_LENGTH` tokens, or tokens outside the passage (offset `None`);
3. the pair with the highest start + end score over all the windows of the question wins;
4. its character offsets give the answer as a substring of the original passage.

The model is loaded from the best checkpoint on Drive, so this part works without re-training.

In [ ]:
import collections
import numpy as np
from transformers import AutoModelForQuestionAnswering, Trainer, TrainingArguments, default_data_collator

best_dir = CHECKPOINT_DIR / ("quick_test" if QUICK_TEST else "run") / "best"
best_model = AutoModelForQuestionAnswering.from_pretrained(str(best_dir)).to(DEVICE)
predictor = Trainer(
    model=best_model,
    args=TrainingArguments(output_dir="/content/tmp_predict", per_device_eval_batch_size=EVAL_BATCH_SIZE,
                           fp16=(DEVICE == "cuda"), report_to="none"),
    data_collator=default_data_collator,
)
print("Model loaded from", best_dir)

def get_logits(feats):
    """Start and end scores for every window (one row per window)."""
    out = predictor.predict(feats.remove_columns(["example_id", "offset_mapping"]))
    start_logits, end_logits = out.predictions
    return start_logits, end_logits

def decode(examples, feats, start_logits, end_logits, n_best=N_BEST, max_answer_length=MAX_ANSWER_LENGTH):
    """Best answer text for each question: {question id: answer}."""
    windows_of = collections.defaultdict(list)  # question id -> its windows
    for i, ex_id in enumerate(feats["example_id"]):
        windows_of[ex_id].append(i)
    all_offsets = feats["offset_mapping"]

    preds = {}
    for ex in examples:
        best_text, best_score = "", -np.inf
        for i in windows_of[ex["id"]]:
            s_log, e_log, offsets = start_logits[i], end_logits[i], all_offsets[i]
            for s in np.argsort(s_log)[-n_best:][::-1]:
                for e in np.argsort(e_log)[-n_best:][::-1]:
                    if offsets[s] is None or offsets[e] is None:  # not in the passage
                        continue
                    if e < s or e - s + 1 > max_answer_length:
                        continue
                    score = s_log[s] + e_log[e]
                    if score > best_score:
                        best_score = score
                        best_text = ex["context"][offsets[s][0]:offsets[e][1]]
        preds[ex["id"]] = best_text
    return preds

# Test on a few validation questions
sample = val_ds.select(range(10))
sample_feats = sample.map(prepare_eval_features, batched=True, remove_columns=val_ds.column_names)
s_log, e_log = get_logits(sample_feats)
sample_preds = decode(sample, sample_feats, s_log, e_log)
for ex in sample:
    print(f"Q: {ex['question']}\n   gold: {ex['answers']['text'][0]!r}   pred: {sample_preds[ex['id']]!r}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded from /content/drive/MyDrive/hlt-qa/checkpoints/roberta/quick_test/best


Q: What happened to Joseph I in 1758?
   gold: 'wounded in an attempted assassination'   pred: 'wounded in an attempted assassination'
Q: Who was implicated in the attempted assassination of Joseph I?
   gold: 'The Távora family and the Duke of Aveiro'   pred: 'the Duke of Aveiro'
Q: Who was expelled from the country after the assassination attempt on Joseph I?
   gold: 'The Jesuits'   pred: 'Jesuits'
Q: In what year did Joseph I make his minister the Count of Oeiras?
   gold: '1759'   pred: '1759'
Q: What act finally broke the power of the aristocracy?
   gold: 'Sebastião de Melo prosecuted every person involved, even women and children'   pred: 'This was the final stroke'
Q: In which year was the Count of Oeiras made the Marquis of Pambal?
   gold: '1770'   pred: '1770'
Q: How long did the Marquis of Pombal rule Portugal?
   gold: "until Joseph I's death in 1779"   pred: '1770'
Q: What did Pombal's enlightenment promote at the expense of individual liberty?
   gold: 'autocracy'   pre

## 7. Evaluation
Quick check on the **validation** set only, with the project metrics (EM, F1, containment: for each question the best score over its gold answers, then the mean). It is used to see that the model is reasonable; the official metrics on the dev set are computed in `04_automatic_metrics`, not here.

In [ ]:
import re
import string
import collections

PUNCT = set(string.punctuation)

def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in PUNCT)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, ref):
    return float(normalize_answer(pred) == normalize_answer(ref))

def f1_score(pred, ref):
    pred_tokens = normalize_answer(pred).split()
    ref_tokens = normalize_answer(ref).split()
    common = collections.Counter(pred_tokens) & collections.Counter(ref_tokens)
    n_same = sum(common.values())
    if n_same == 0:
        return 0.0
    precision = n_same / len(pred_tokens)
    recall = n_same / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

def containment(pred, ref):
    pred_tokens = normalize_answer(pred).split()
    ref_tokens = normalize_answer(ref).split()
    if not ref_tokens:  # case of no reference after normalization
        return exact_match(pred, ref)
    n = len(ref_tokens)
    for i in range(len(pred_tokens) - n + 1):
        if pred_tokens[i:i + n] == ref_tokens:
            return 1.0
    return 0.0

def score_prediction(pred, refs):
    """Metrics of one prediction: best score over all its references."""
    return {
        "em": max(exact_match(pred, r) for r in refs),
        "f1": max(f1_score(pred, r) for r in refs),
        "containment": max(containment(pred, r) for r in refs),
    }

print(score_prediction("Normandy is located in France.", ["France"]))  # dummy sample

# Validation
eval_ds = val_ds.select(range(500)) if QUICK_TEST else val_ds
eval_ids = set(eval_ds["id"])
eval_feats = val_eval_feats.filter(lambda f: f["example_id"] in eval_ids) if QUICK_TEST else val_eval_feats

val_start, val_end = get_logits(eval_feats)
val_preds = decode(eval_ds, eval_feats, val_start, val_end)

val_scores = [score_prediction(val_preds[ex["id"]], ex["answers"]["text"]) for ex in eval_ds]
means = {m: 100 * np.mean([s[m] for s in val_scores]) for m in ["em", "f1", "containment"]}
print(f"Validation ({len(eval_ds)} questions): "
      f"EM = {means['em']:.2f}  F1 = {means['f1']:.2f}  containment = {means['containment']:.2f}")

{'em': 0.0, 'f1': 0.33333333333333337, 'containment': 1.0}


Validation (500 questions): EM = 45.20  F1 = 62.83  containment = 58.40


## 8. Predictions
Predict on the dev set (our test set) and write one JSON line per question with the question id, the predicted text and the gold answers. With `QUICK_TEST = True` only 200 dev questions are predicted, in a separate file, so the real `OUTPUT_FILE` is never overwritten by a test.

In [ ]:
import json

pred_ds = dev_ds.select(range(200)) if QUICK_TEST else dev_ds
pred_feats = pred_ds.map(prepare_eval_features, batched=True, remove_columns=dev_ds.column_names)

dev_start, dev_end = get_logits(pred_feats)
dev_preds = decode(pred_ds, pred_feats, dev_start, dev_end)

out_file = PREDICTIONS_DIR / f"{SYSTEM_ID}_quick_test.jsonl" if QUICK_TEST else OUTPUT_FILE
with open(out_file, "w", encoding="utf-8") as f:
    for ex in pred_ds:
        row = {"id": ex["id"], "text": dev_preds[ex["id"]], "references": ex["answers"]["text"]}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Checks: one line per question, no empty answers
lines = out_file.read_text(encoding="utf-8").splitlines()
assert len(lines) == len(pred_ds)
n_empty = sum(1 for l in lines if not json.loads(l)["text"].strip())
print(f"Wrote {len(lines)} predictions to {out_file} (empty answers: {n_empty})")
print(lines[0])

Wrote 200 predictions to /content/drive/MyDrive/hlt-qa/data/predictions/roberta_quick_test.jsonl (empty answers: 0)
{"id": "56be4db0acb8001400a502ec", "text": "Denver Broncos", "references": ["Denver Broncos", "Denver Broncos", "Denver Broncos"]}
